In [1]:
import pandas as pd

# Load the Excel file
file_path = r"C:\Users\scaale00\Documents\GND-Python\data\VERZ_f-Level6-alle-MOCs_20240702_nme.xlsx"
df = pd.read_excel(file_path)

# Define the column name
column_name = "Abweichende Benennungen"

# Function to split and clean name variants
def split_name_variants(cell):
    if pd.isna(cell):
        return []
    return [name.strip() for name in str(cell).split(",") if name.strip()]

# Apply the cleaning function
df["Name Variants List"] = df[column_name].apply(split_name_variants)

# Preview
print(df[["Abweichende Benennungen", "Name Variants List"]].head())

# Optional: Save to a new Excel or CSV
output_path = r"C:\Users\scaale00\Documents\GND-Python\data\cleaned_name_variants.xlsx"
df.to_excel(output_path, index=False)

                             Abweichende Benennungen  \
0                                                NaN   
1                                                NaN   
2                                                NaN   
3  Charles Sanders Peirce Sesquicentennial Intern...   
4                 WAC World Archaeological Congress    

                                  Name Variants List  
0                                                 []  
1                                                 []  
2                                                 []  
3  [Charles Sanders Peirce Sesquicentennial Inter...  
4                [WAC World Archaeological Congress]  


In [2]:
import pandas as pd
import re

# Load the CSV file
csv_path = r"C:\Users\scaale00\Documents\GND-Python\data\gnd_filtered_records.csv"
df = pd.read_csv(csv_path, dtype=str)  # Read as strings to avoid NaN issues

# Define the column name
column_name = "Field 411"

# Function to split and clean name variants from multiple delimiters
def split_variants(text):
    if pd.isna(text):
        return []
    # Split on both commas and pipes
    parts = re.split(r'[|,]', text)
    # Clean and deduplicate
    cleaned = {part.strip() for part in parts if part.strip()}
    return list(cleaned)

# Apply the function to create a clean list of variants
df["Field 411 Variants"] = df[column_name].apply(split_variants)

# Preview
print(df[[column_name, "Field 411 Variants"]].head())

# Optional: Save output
output_csv_path = r"C:\Users\scaale00\Documents\GND-Python\data\gnd_filtered_records_cleaned.csv"
df.to_csv(output_csv_path, index=False)

                                           Field 411  \
0  Conferentie van Niet-Kernwapenstaten | Confere...   
1                                                NaN   
2                                                NaN   
3                                                NaN   
4                                                NaN   

                                  Field 411 Variants  
0  [Conferentie van Niet-Kernwapenstaten, Confere...  
1                                                 []  
2                                                 []  
3                                                 []  
4                                                 []  


In [3]:
import pandas as pd

# Load cleaned Excel and CSV
excel_path = r"C:\Users\scaale00\Documents\GND-Python\data\cleaned_name_variants.xlsx"
csv_path = r"C:\Users\scaale00\Documents\GND-Python\data\gnd_filtered_records_cleaned.csv"

df_excel = pd.read_excel(excel_path)
df_csv = pd.read_csv(csv_path, dtype=str)

# Fill NAs
df_csv["Field 111"] = df_csv["Field 111"].fillna("")
df_csv["Field 411 Variants"] = df_csv["Field 411 Variants"].apply(eval)  # Convert stringified list back to list

# Prepare match results
matches = []

for idx, row in df_excel.iterrows():
    title = row["title"]
    variants = row["Name Variants List"]
    
    match = None

    # Step 1: Try exact match on Field 111
    if title in df_csv["Field 111"].values:
        match = df_csv[df_csv["Field 111"] == title].iloc[0].to_dict()
        match_type = "title → Field 111"

    # Step 2: Try variant in Field 111
    elif any(variant in df_csv["Field 111"].values for variant in variants):
        for variant in variants:
            match_row = df_csv[df_csv["Field 111"] == variant]
            if not match_row.empty:
                match = match_row.iloc[0].to_dict()
                match_type = "variant → Field 111"
                break

    # Step 3: Try variant in Field 411 Variants
    else:
        for i, csv_row in df_csv.iterrows():
            field_411_list = csv_row["Field 411 Variants"]
            if any(variant in field_411_list for variant in variants):
                match = csv_row.to_dict()
                match_type = "variant → Field 411"
                break

    # Record the match
    matches.append({
        "title": title,
        "match_type": match_type if match else "no match",
        "matched_row": match
    })

# Turn into dataframe
results_df = pd.DataFrame(matches)

# You can now flatten matched_row into columns if you want

# Optional: save
results_df.to_excel(r"C:\Users\scaale00\Documents\GND-Python\data\matched_results.xlsx", index=False)

KeyboardInterrupt: 